# Analyze BEAST per-frame latents (pair with keypoints)

Validate that the unsupervised BEAST latents from
`behavior_784803_2025-07-03_13-55-13/.../bottom_camera.mp4` are meaningful and
pair usefully with Lightning Pose keypoint tracking, before scaling to a
multi-session backbone.

Pipeline recap: decimate video -> `beast extract` -> `beast train` (ResNet-AE,
12 latents) -> `beast predict` -> per-frame latents `bottom_camera.npy`
shape `(1899809, 12)`.

Run top-to-bottom in a JupyterLab Cloud Workstation with the latents +
keypoint assets attached under `/data`. Linear and minimal by design.

## Setup

Imports and file discovery. Paths are globbed under `/data` so the notebook is robust to exact asset mount names.

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "/data"
EXPECTED_FRAMES = 1_899_809
NUM_LATENTS = 12


def find_one(patterns, what):
    """Return the single matching path, or raise with a helpful listing."""
    hits = []
    for pat in patterns:
        hits.extend(glob.glob(pat, recursive=True))
    hits = sorted(set(hits))
    if not hits:
        listing = "\n".join(sorted(glob.glob(os.path.join(DATA_DIR, "*")))) or "(empty)"
        raise FileNotFoundError(
            f"No {what} found for patterns {patterns}.\n/data contains:\n{listing}"
        )
    if len(hits) > 1:
        print(f"[warn] multiple {what} candidates; using first:")
        for h in hits:
            print("   ", h)
    return hits[0]


## 1. Load latents

Find `bottom_camera.npy` (the captured beast-output asset) and assert its shape/dtype.

In [ ]:
npy_path = find_one(
    [
        os.path.join(DATA_DIR, "**", "bottom_camera.npy"),
        os.path.join(DATA_DIR, "**", "*latents*.npy"),
    ],
    "latents .npy",
)
print("latents:", npy_path)

z = np.load(npy_path)
print("shape:", z.shape, "dtype:", z.dtype)

assert z.ndim == 2, f"expected 2-D latents, got {z.shape}"
assert z.shape[1] == NUM_LATENTS, f"expected {NUM_LATENTS} latents, got {z.shape[1]}"
if z.shape[0] != EXPECTED_FRAMES:
    print(f"[warn] frame count {z.shape[0]} != expected {EXPECTED_FRAMES}")
z = z.astype(np.float32, copy=False)
N_FRAMES = z.shape[0]


## 2. Latent sanity check (the key gate)

Per-dim mean/std should show **real spread**, not the ~0.001 collapse seen in
the 2-epoch smoke test. Quick histograms + a short-window trace of a few dims.

In [ ]:
stats = pd.DataFrame({
    "mean": z.mean(axis=0),
    "std": z.std(axis=0),
    "min": z.min(axis=0),
    "max": z.max(axis=0),
}, index=[f"z{i}" for i in range(NUM_LATENTS)])
stats.loc["__all__"] = [z.mean(), z.std(), z.min(), z.max()]
print(stats.round(4))

collapsed = (stats["std"].iloc[:NUM_LATENTS] < 1e-2).sum()
print(f"\ndims with std < 1e-2 (near-collapse): {collapsed} / {NUM_LATENTS}")
assert collapsed < NUM_LATENTS, "ALL latent dims collapsed -- training failed."


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 8), sharex=False)
for i, ax in enumerate(axes.ravel()):
    ax.hist(z[:, i], bins=80, color="steelblue")
    ax.set_title(f"z{i}  (std={z[:, i].std():.3f})", fontsize=9)
fig.suptitle("Per-dim latent distributions (all frames)")
fig.tight_layout()
plt.show()


In [ ]:
WIN = slice(0, 2000)  # short window of frames
fig, ax = plt.subplots(figsize=(14, 5))
for i in range(NUM_LATENTS):
    ax.plot(np.arange(WIN.start, WIN.stop), z[WIN, i], lw=0.8, label=f"z{i}")
ax.set_xlabel("frame")
ax.set_ylabel("latent value")
ax.set_title(f"Latent traces over frames {WIN.start}:{WIN.stop}")
ax.legend(ncol=6, fontsize=8, loc="upper right")
fig.tight_layout()
plt.show()


## 3. Load Lightning Pose keypoints

DLC/LP-style CSV under `.../intermediate_data`: multi-row header
(scorer / bodypart / coord), one row per video frame. The loader handles a
2- or 3-row header and returns per-keypoint `x, y, likelihood`.

In [ ]:
csv_path = find_one(
    [
        os.path.join(DATA_DIR, "keypoint_tracking_bottomview*", "**",
                     "intermediate_data", "*.csv"),
        os.path.join(DATA_DIR, "**", "intermediate_data", "*.csv"),
        os.path.join(DATA_DIR, "**", "*bottom*.csv"),
    ],
    "Lightning Pose CSV",
)
print("keypoints:", csv_path)


def load_lp_csv(path):
    """Load a DLC/LP CSV. Returns (df, bodyparts). Columns are MultiIndex
    (bodypart, coord) with coord in {x, y, likelihood}."""
    for header in ([0, 1, 2], [0, 1]):
        try:
            df = pd.read_csv(path, header=header, index_col=0)
        except Exception:
            continue
        cols = df.columns
        # Drop the scorer level if present (3-row header) so we are left with
        # (bodypart, coord).
        if cols.nlevels == 3:
            df.columns = cols.droplevel(0)
        if df.columns.nlevels == 2 and {"x", "y"}.issubset(
            set(df.columns.get_level_values(1))
        ):
            bodyparts = list(dict.fromkeys(df.columns.get_level_values(0)))
            return df, bodyparts
    raise ValueError(f"Could not parse LP CSV header: {path}")


kp, BODYPARTS = load_lp_csv(csv_path)
print("rows:", len(kp), " bodyparts:", BODYPARTS)
kp.head()


## 4. Align latents <-> keypoints

Row `i` of the latents should correspond to frame `i` of the keypoints. Assert
equal length; if they differ, truncate to the common length and flag it (e.g.
if LP ran on a different frame set).

In [ ]:
n_kp = len(kp)
print(f"latent frames: {N_FRAMES}   keypoint frames: {n_kp}")

if n_kp != N_FRAMES:
    n = min(n_kp, N_FRAMES)
    print(f"[warn] length mismatch ({n_kp} vs {N_FRAMES}); truncating both to {n}.")
    z_a = z[:n]
    kp_a = kp.iloc[:n]
else:
    print("OK: keypoint row count == latent row count.")
    z_a = z
    kp_a = kp
N = z_a.shape[0]


## 5. Basic latent <-> keypoint relationship (the core test)

Per-frame keypoint speed `v = sqrt(dx^2 + dy^2)`, then compare latents against
that speed: overlaid traces, a correlation matrix, and an optional linear fit.

In [ ]:
# Pick a key bodypart; prefer a tongue/jaw keypoint if present.
PREFERRED = ["tongue_tip_center", "tongue_tip", "tongue", "jaw"]
KP_NAME = next((b for b in PREFERRED if b in BODYPARTS), BODYPARTS[0])
print("using bodypart:", KP_NAME)

x = kp_a[(KP_NAME, "x")].to_numpy(dtype=np.float32)
y = kp_a[(KP_NAME, "y")].to_numpy(dtype=np.float32)
if (KP_NAME, "likelihood") in kp_a.columns:
    lik = kp_a[(KP_NAME, "likelihood")].to_numpy(dtype=np.float32)
else:
    lik = np.ones(N, dtype=np.float32)

# Per-frame speed (pad first frame so it stays length N), mirroring kinematics.
speed = np.empty(N, dtype=np.float32)
speed[0] = 0.0
speed[1:] = np.sqrt(np.diff(x) ** 2 + np.diff(y) ** 2)
print(f"speed: mean={np.nanmean(speed):.3f} max={np.nanmax(speed):.3f} "
      f"frac_nan={np.isnan(speed).mean():.3f}")


In [ ]:
# Overlay a few latent dims against keypoint speed over a short window.
W = slice(0, 2000)
fig, (a0, a1) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for i in range(NUM_LATENTS):
    a0.plot(np.arange(W.start, W.stop), z_a[W, i], lw=0.7)
a0.set_ylabel("latents")
a0.set_title(f"Latents vs {KP_NAME} speed, frames {W.start}:{W.stop}")
a1.plot(np.arange(W.start, W.stop), speed[W], color="crimson", lw=0.9)
a1.set_ylabel(f"{KP_NAME} speed")
a1.set_xlabel("frame")
fig.tight_layout()
plt.show()


In [ ]:
# Correlation matrix: 12 latent dims vs {x, y, speed} of the chosen keypoint.
feat = pd.DataFrame({f"z{i}": z_a[:, i] for i in range(NUM_LATENTS)})
feat[f"{KP_NAME}_x"] = x
feat[f"{KP_NAME}_y"] = y
feat[f"{KP_NAME}_speed"] = speed

# Restrict to confidently-tracked frames to avoid LP dropout artefacts.
mask = np.isfinite(speed) & (lik > 0.9)
print(f"frames used (likelihood>0.9 & finite): {mask.sum()} / {N}")
corr = feat[mask].corr()

kp_cols = [f"{KP_NAME}_x", f"{KP_NAME}_y", f"{KP_NAME}_speed"]
sub = corr.loc[[f"z{i}" for i in range(NUM_LATENTS)], kp_cols]
fig, ax = plt.subplots(figsize=(5, 7))
im = ax.imshow(sub.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(kp_cols)), kp_cols, rotation=45, ha="right")
ax.set_yticks(range(NUM_LATENTS), [f"z{i}" for i in range(NUM_LATENTS)])
for (r, c), val in np.ndenumerate(sub.values):
    ax.text(c, r, f"{val:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("Latent <-> keypoint correlation")
fig.tight_layout()
plt.show()

print("\nmax |corr| with speed:", sub[f"{KP_NAME}_speed"].abs().max().round(3))


In [ ]:
# (Optional) linear fit: latents -> keypoint x and y; report R^2.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

Xz = z_a[mask]
for target, name in [(x[mask], "x"), (y[mask], "y"), (speed[mask], "speed")]:
    reg = LinearRegression().fit(Xz, target)
    r2 = r2_score(target, reg.predict(Xz))
    print(f"latents -> {KP_NAME}_{name}:  R^2 = {r2:.3f}")


## 6. (Optional, light) Latent structure

PCA variance of the 12 dims, and a 2-D scatter of a ~20k-frame subsample
colored by keypoint speed. UMAP skipped this pass to avoid a new dependency.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(z_a)
evr = pca.explained_variance_ratio_
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, NUM_LATENTS + 1), evr, color="slateblue")
ax.plot(range(1, NUM_LATENTS + 1), np.cumsum(evr), "-o", color="black",
        label="cumulative")
ax.set_xlabel("PC")
ax.set_ylabel("explained variance ratio")
ax.set_title("PCA of the 12 latent dims")
ax.legend()
fig.tight_layout()
plt.show()
print("cumulative EVR:", np.cumsum(evr).round(3))


In [ ]:
# 2-D PCA scatter of a subsample, colored by keypoint speed.
rng = np.random.default_rng(0)
idx = np.where(mask)[0]
sub_idx = rng.choice(idx, size=min(20_000, idx.size), replace=False)
proj = pca.transform(z_a[sub_idx])[:, :2]
c = np.clip(speed[sub_idx], 0, np.nanpercentile(speed[sub_idx], 99))

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(proj[:, 0], proj[:, 1], c=c, s=4, cmap="viridis", alpha=0.6)
fig.colorbar(sc, ax=ax, label=f"{KP_NAME} speed")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title(f"Latent PCA subsample (n={sub_idx.size}) colored by speed")
fig.tight_layout()
plt.show()


## Summary / gates

- **Latent gate:** per-dim std clearly non-zero / varied (latents not collapsed);
  PCA shows the 12 dims carry structure.
- **Keypoint gate:** frame alignment succeeded and at least one latent<->keypoint
  relationship is quantified (correlation and regression R^2 above).

If both gates pass, the latents are meaningful enough to proceed toward the
multi-session backbone (see PLAN.md roadmap).